In [1]:
import numpy as np
import pandas as pd

def logistic_curve(x, r):
    return r * x * (1 - x)

r_value = 2.5

# Values used to evaluate the rule
x_values = np.linspace(0, 1, 100)

# Initial value for iteration
x_it = 0.1

rows = []

for t, x_t in enumerate(x_values):

    # Apply the rule to x_t
    x_next = logistic_curve(x_t, r_value)

    # Apply the same rule to the iterated value
    x_it_next = logistic_curve(x_it, r_value)

    rows.append([t, x_t, x_next, x_it, x_it_next])

    # Output becomes the input for the next iteration
    x_it = x_it_next


df_iteration = pd.DataFrame(
    rows,
    columns=['t', 'x_t', 'x_next', 'x_it', 'x_it_next']
)


# Color the two processes
df_iteration.style \
    .set_properties(
        subset=['x_t', 'x_next'],
        **{'background-color': 'lightblue'}
    ) \
    .set_properties(
        subset=['x_it', 'x_it_next'],
        **{'background-color': 'lightyellow'}
    ) \
    .format(precision=4)

,t,x_t,x_next,x_it,x_it_next
0,0,0.0000,0.0000,0.1000,0.2250
1,1,0.0101,0.0250,0.2250,0.4359
2,2,0.0202,0.0495,0.4359,0.6147
3,3,0.0303,0.0735,0.6147,0.5921
4,4,0.0404,0.0969,0.5921,0.6038
5,5,0.0505,0.1199,0.6038,0.5981
6,6,0.0606,0.1423,0.5981,0.6010
7,7,0.0707,0.1643,0.6010,0.5995
8,8,0.0808,0.1857,0.5995,0.6002
9,9,0.0909,0.2066,0.6002,0.5999


In [2]:
# Detect when the system reaches the fixed-point attractor

df_plot = df_iteration.copy()

tolerance = 1e-6

# How much does the state change at each iteration?
df_plot['change'] = abs(
    df_plot['x_it_next'] - df_plot['x_it']
)

# Find the first t from which all later changes
# remain below the tolerance
t_attractor = None

for t in df_plot['t']:

    remaining = df_plot.loc[
        df_plot['t'] >= t,
        'change'
    ]

    if (remaining < tolerance).all():
        t_attractor = t
        break


# Identify the three stages
df_plot['stage'] = 'Transient'

df_plot.loc[
    df_plot['t'] == 0,
    'stage'
] = 'Initial condition'

df_plot.loc[
    df_plot['t'] >= t_attractor,
    'stage'
] = 'Attractor'


df_plot[['t', 'x_it', 'x_it_next', 'change', 'stage']].head(30)

,t,x_it,x_it_next,change,stage
0,0,0.100000,0.225000,1.250000e-01,Initial condition
1,1,0.225000,0.435938,2.109375e-01,Transient
2,2,0.435938,0.614740,1.788025e-01,Transient
3,3,0.614740,0.592087,2.265315e-02,Transient
4,4,0.592087,0.603800,1.171320e-02,Transient
5,5,0.603800,0.598064,5.736155e-03,Transient
6,6,0.598064,0.600959,2.894807e-03,Transient
7,7,0.600959,0.599518,1.440330e-03,Transient
8,8,0.599518,0.600240,7.218826e-04,Transient
9,9,0.600240,0.599880,3.605057e-04,Transient


In [ ]:
import altair as alt

chart = alt.Chart(df_plot).mark_point(
    filled=True,
    size=90
).encode(
    x=alt.X(
        't:Q',
        title='Iteration'
    ),
    y=alt.Y(
        'x_it:Q',
        title='x'
    ),
    color=alt.Color(
        'stage:N',
        title=None,
        scale=alt.Scale(
            domain=[
                'Initial condition',
                'Transient',
                'Attractor'
            ],
            range=[
                'black',
                'orange',
                'blue'
            ]
        )
    ),
    tooltip=[
        alt.Tooltip('t:Q', title='Iteration'),
        alt.Tooltip('x_it:Q', title='x', format='.6f'),
        alt.Tooltip('stage:N', title='Stage')
    ]
).properties(
    width=750,
    height=380,
    title='From Initial Condition to Attractor'
).interactive()

chart